# commitNK: 最適コミットメント政策と図7.4 の再現

教科書 7.6 節のアルゴリズムに従い、過去のラグランジュ乗数 (φ_EE,-1, φ_PC,-1) を内生状態とする時間反復法で
最適コミットメント政策を解く。同じカリブレーションで最適裁量政策も解いて比較する。

**設定**: 2状態モデル (s_H, s_L)、自然利子率ショックのみ。
決定論的なショック系列 (t=1..8 が L、t≥9 が H) に対して両政策の経路を描き、
教科書 図7.4 を再現する。

## モジュール構成
- `src/nk_model.jl` — `Model` 構造体
- `src/shocks.jl` — `build_two_state_chain` で 2 状態 Markov 連鎖を構成
- `src/discretionary.jl` — 裁量政策の時間反復解法
- `src/commitment.jl` — コミット政策の時間反復解法 (本ノートの中心)
- `src/simulation.jl` — 決定論的経路シミュレーション

## モデル方程式 (一般 σ, discNK 規約のショック)

- IS: $y = E[y'] - \sigma(r_n - E[\pi'] ) + g$
- PC: $\pi = \kappa y + \beta E[\pi']$
- FOC π: $\pi + \phi_{PC} - \phi_{PC,-1} - \sigma \beta^{-1} \phi_{EE,-1} = 0$
- FOC y: $\lambda y + \phi_{EE} - \beta^{-1}\phi_{EE,-1} - \kappa \phi_{PC} = 0$
- ZLB: $r_n \ge 0$, $\phi_{EE} \ge 0$, 相補スラックネス

## アルゴリズム要約

各反復 n→n+1 で、グリッド点 $(\phi_{EE,-1,k}, \phi_{PC,-1,l}, s_i)$ ごとに:
1. **Case A** (ZLB 非バインド, $\phi_{EE} = 0$): φ_PC についての 1 次元非線形方程式を NLsolve で解き、IS から $r_n$ を計算。$r_n \ge 0$ なら採用。
2. **Case B** (ZLB バインド, $r_n = 0$): $(\phi_{EE}, \phi_{PC})$ についての 2 次元非線形系を NLsolve で解く。

期待値 $y_e, \pi_e$ は古い政策関数 $\varsigma^{(n-1)}$ を 2 次元線形補間して計算。

In [1]:
using Plots
using LaTeXStrings

include("src/nk_model.jl")
include("src/shocks.jl")
include("src/discretionary.jl")
include("src/commitment.jl")
include("src/simulation.jl");

## カリブレーション

教科書 7.2.1 の TwostateNK で校正された値 (κ ≈ 0.0067) を使い、σ=1 (textbook 7.6 規約)。
λ は forward guidance の強さに大きく影響する。ここでは λ=0.3 として、
図7.4 と定性的に整合する経路を生成する。

ショックの値 sL は、裁量政策のもとで危機時の出力ギャップが約 -7 となるように選ぶ。

In [ ]:
# 2状態モデル用のカリブレーション
# 教科書 7.6 節は σ=1 を暗黙に用いた式 (FOC π に σ が現れない形) を提示している。
# このため σ=1 を用いる (TwostateNK / 7.6節の規約)。
# κ, λ は教科書には明示されていないため、図 7.4 と整合的な値を試行錯誤的に選んだ。
rstar = 0.75            # 定常名目金利 (% / 四半期, 年率 3%)
bet   = 1/(1+rstar/100) # 割引率
sig   = 1.0             # 教科書 7.6 規約 (σ=1)
kap   = 0.009           # フィリップス曲線の傾き (disc π_L ≈ -1% 年率となるよう選定)
lam   = 0.1             # 損失関数における y の重み (forward guidance を生む大きさに調整)

# Model 構造体に直接渡す (alp, the, ome, sigu などは未使用)
m = Model(rstar, bet, sig, 0.66, 7.66, 0.47, kap, lam,
          0.0, 0.0, 0.0, 0.0, 1, 1, 5000, 1e-7)
println("σ=$(m.sig), κ=$(m.kap), λ=$(m.lam), β=$(round(m.bet, digits=4)), rstar=$(m.rstar)")

In [ ]:
# 2状態 Markov 連鎖 (s_H, s_L)
# 注: 教科書 7.4 節は p_H = 0 (H は吸収状態) を明示している。これにより、状態 H に戻った時点で
# 将来の不確実性の影響はなくなり、disc 政策の H 状態が定常状態 (y=0, π=0, r=rstar) に一致する。
sH = m.rstar          # 平常時 (steady-state natural rate)
sL = -1.58            # 危機時。裁量政策で y_L ≈ -7, π_L_ann ≈ -1 となるよう選定
pH = 0.0              # 平常 → 危機への遷移確率 (H は吸収)
pL = 0.75             # 危機が継続する確率 (平均危機期間 = 1/(1-pL) = 4 期)
Gs1d, Ps = build_two_state_chain(sH, sL, pH, pL)
println("sH=$sH, sL=$sL, pH=$pH, pL=$pL")
println("Ps =\n$Ps")

## 1. 最適裁量政策

`solve_discretionary` は `Gs::Ns×2` 行列 (列1=g, 列2=u) を期待するので、u=0 のダミー列を付与。

In [4]:
Gs_disc = hcat(Gs1d, zeros(2))
@time yvec_d, pvec_d, rvec_d, iter_d = solve_discretionary(m; Gs=Gs_disc, Ps=Ps)
println("Discretionary: $(iter_d) iter")
println("  state H: y=$(round(yvec_d[1], digits=3)), π_ann=$(round(4*pvec_d[1], digits=3)), r_ann=$(round(4*rvec_d[1], digits=3))")
println("  state L: y=$(round(yvec_d[2], digits=3)), π_ann=$(round(4*pvec_d[2], digits=3)), r_ann=$(round(4*rvec_d[2], digits=3))")

  0.079417 seconds (399.17 k allocations: 19.631 MiB, 99.83% compilation time)
Discretionary: 78 iter
  state H: y=-0.0, π_ann=0.0, r_ann=3.0
  state L: y=-7.033, π_ann=-0.737, r_ann=0.0


## 2. 最適コミットメント政策

内生状態のグリッドを設定して `solve_commitment` を実行。

In [ ]:
# φ_EE は ZLB の乗数 (≥0)、φ_PC はインフレターゲットの乗数 (符号自由)
# 注意: グリッドの範囲は重要。シミュレーションで実現する範囲を必ずカバーすること。
# 特に φ_PC は危機が長引くと正に大きく蓄積する (forward guidance を生むメカニズム)。
# 範囲外で extrapolate すると、政策関数が正確に評価できなくなる。
grid_φ_EE = collect(range(0.0, 30.0, length=31))    # 0..30, step=1
grid_φ_PC = collect(range(-5.0, 30.0, length=71))   # -5..30, step=0.5
println("grid sizes: N_EE=$(length(grid_φ_EE)) (max=$(maximum(grid_φ_EE))), N_PC=$(length(grid_φ_PC)) (range=[$(minimum(grid_φ_PC)), $(maximum(grid_φ_PC))]), Ns=$(length(Gs1d))")
@time commit = solve_commitment(m; Gs=Gs1d, Ps=Ps,
                                 grid_φ_EE=grid_φ_EE, grid_φ_PC=grid_φ_PC,
                                 verbose=true, tol=1e-6, max_outer=600)
println("Commitment converged in $(commit.iter) iter (final diff=$(commit.diff))")

In [6]:
# 定常状態 (φ_EE=0, φ_PC=0, s=H) での解が ≈ (0, 0, rstar) であることを確認
ik0 = 1                           # φ_EE = 0
il0 = findfirst(x -> x ≈ 0.0, grid_φ_PC)  # φ_PC = 0
println("政策関数の値 at (φ_EE=0, φ_PC=0):")
println("  s=H: y=$(round(commit.y[ik0,il0,1],digits=3)), π_ann=$(round(4*commit.π[ik0,il0,1],digits=3)), r_ann=$(round(4*commit.r_n[ik0,il0,1],digits=3))")
println("  s=L: y=$(round(commit.y[ik0,il0,2],digits=3)), π_ann=$(round(4*commit.π[ik0,il0,2],digits=3)), r_ann=$(round(4*commit.r_n[ik0,il0,2],digits=3))")

政策関数の値 at (φ_EE=0, φ_PC=0):
  s=H: y=0.0, π_ann=0.0, r_ann=3.0
  s=L: y=-3.648, π_ann=0.11, r_ann=0.0


## 3. 決定論的シミュレーション (図7.4 再現)

ショック系列: t = 1..8 が L、t ≥ 9 が H。初期内生状態 $(\phi_{EE,-1}, \phi_{PC,-1}) = (0, 0)$。

In [7]:
T = 20
shock_path = crisis_shock_path(T, 9; H_idx=1, L_idx=2)
sim_d = simulate_discretionary(yvec_d, pvec_d, rvec_d, shock_path)
sim_c = simulate_commitment(commit, shock_path; φ_EE0=0.0, φ_PC0=0.0)

println("shock path (1=H, 2=L): ", shock_path)
println("\ny commit: ", round.(sim_c.y, digits=3))
println("y disc:   ", round.(sim_d.y, digits=3))
println("\nπ commit (annualized): ", round.(4 .* sim_c.π, digits=3))
println("π disc (annualized):   ", round.(4 .* sim_d.π, digits=3))
println("\nr_n commit (annualized): ", round.(4 .* sim_c.r_n, digits=3))
println("r_n disc (annualized):   ", round.(4 .* sim_d.r_n, digits=3))

shock path (1=H, 2=L): [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

y commit: [-3.648, -3.069, -2.558, -2.114, -1.737, -1.43, -1.164, -0.945, 4.333, 3.517, 2.755, 2.163, 1.704, 1.348, 1.071, 0.856, 0.69, 0.56, 0.46, 0.381]
y disc:   [-7.033, -7.033, -7.033, -7.033, -7.033, -7.033, -7.033, -7.033, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0]

π commit (annualized): [0.11, 0.233, 0.349, 0.453, 0.535, 0.592, 0.641, 0.684, 0.696, 0.584, 0.494, 0.424, 0.37, 0.327, 0.294, 0.269, 0.249, 0.234, 0.222, 0.213]
π disc (annualized):   [-0.737, -0.737, -0.737, -0.737, -0.737, -0.737, -0.737, -0.737, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

r_n commit (annualized): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.446, 1.056, 1.53, 1.898, 2.184, 2.405, 2.578, 2.711, 2.815, 2.895, 2.958]
r_n disc (annualized):   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]


In [ ]:
ts = 1:T
p1 = plot(ts, sim_c.y, label="コミットメント政策", color=:black, linestyle=:solid, linewidth=2,
          xlabel="", ylabel="", title=L"産出ギャップ, y", legend=false)
plot!(p1, ts, sim_d.y, label="裁量政策", color=:gray, linestyle=:dash, linewidth=2)

p2 = plot(ts, 4 .* sim_c.π, label="コミットメント政策", color=:black, linestyle=:solid, linewidth=2,
          xlabel="", ylabel="", title=L"インフレ率, \pi", legend=:bottomright, legendfontsize=8)
plot!(p2, ts, 4 .* sim_d.π, label="裁量政策", color=:gray, linestyle=:dash, linewidth=2)

p3 = plot(ts, 4 .* sim_c.r_n, label="コミットメント政策", color=:black, linestyle=:solid, linewidth=2,
          xlabel="", ylabel="", title=L"政策金利, r_n", legend=false)
plot!(p3, ts, 4 .* sim_d.r_n, label="裁量政策", color=:gray, linestyle=:dash, linewidth=2)

plot(p1, p2, p3, layout=(1,3), size=(1100, 320),
     plot_title="図7.4 (再現): 2状態モデルにおける最適コミットメント政策と最適裁量政策",
     plot_titlefontsize=11)

## 結果の解釈

- **裁量政策** (破線): 危機状態 L にいる間 (t=1..8) は、状態のみに依存するので毎期同じ値を取り、ZLB が拘束 (r_n=0)。t=9 で平常状態 H に戻ると一気に定常状態水準 (y=0, π=0, r_n≈3% 年率) に戻る。教科書の説明: 「政策金利をゼロより下げることができないため、産出ギャップ、インフレ率ともに大きく落ち込む」。

- **コミットメント政策** (実線): 過去のラグランジュ乗数 $(\phi_{EE,-1}, \phi_{PC,-1})$ を内生状態として持ち続けるため、危機状態が長引くほど将来の追加金融緩和を約束 (forward guidance)。教科書の説明: 「最初の8期間が過ぎた後の状態 H のもとでも、ゼロ金利をしばらく続けることで産出ギャップとインフレ率はゼロを超えてオーバーシュートする、そのような政策にコミットする」。
    - 危機中の y, π の落ち込みは裁量より緩やか (時間とともに乗数が蓄積し、期待インフレ・期待出力ギャップが上振れしていく効果)。
    - 危機脱出後 (t≥9) も累積した乗数の影響で出力ギャップが正に振れ、徐々に定常状態へ収束。
    - 政策金利は危機脱出直後 (t=9) も ZLB のままで、その後数期間にわたってゆっくりと立ち上がる。教科書の解説どおりの forward guidance パターン。

## 実装上の注意点 (デバッグメモ)

1. **2 次元グリッドの範囲**: コミット政策では危機が長引くと内生状態 $\phi_{PC}$ が大きく蓄積する (ピーク値が 5〜20 程度になりうる)。グリッドが狭いと範囲外で外挿が起きて政策関数が歪み、forward guidance が極端に弱くなる。`grid_φ_PC` は十分広く取る必要がある。

2. **σ の取り扱い**: 教科書 7.6 節は σ=1 を暗黙に用いる。discNK のように σ=6.25 にすると、2状態モデルでは裁量政策がデフレスパイラル (発散) に陥るため、σ=1 が適切。

3. **κ, λ の感応度**:
    - κ²/λ が小さいと内生状態の固有値が 1 に近づき、収束が極端に遅くなる (`y, π → 0` までに何十期もかかる)。
    - κ が大きすぎると π のオーバーシュートが過大になる。
    - 教科書のグラフと整合的にするためには κ=0.009, λ=0.1 程度の組み合わせがよい。